In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lokeshhate/zomato-dataset/Dataset .csv


In [5]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("/kaggle/input/datasets/lokeshhate/zomato-dataset/Dataset .csv")

df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [8]:
#Data Preprocessing
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df['City'] = df['City'].fillna('Unknown')
df['Average Cost for two'] = df['Average Cost for two'].fillna(df['Average Cost for two'].median())

df = df.drop_duplicates()

In [9]:
#Select Important Features
df = df[['Restaurant Name',
         'City',
         'Cuisines',
         'Price range',
         'Average Cost for two',
         'Aggregate rating']]

df.head()

,Restaurant Name,City,Cuisines,Price range,Average Cost for two,Aggregate rating
0,Le Petit Souffle,Makati City,"French, Japanese, Desserts",3,1100,4.8
1,Izakaya Kikufuji,Makati City,Japanese,3,1200,4.5
2,Heat - Edsa Shangri-La,Mandaluyong City,"Seafood, Asian, Filipino, Indian",4,4000,4.4
3,Ooma,Mandaluyong City,"Japanese, Sushi",4,1500,4.9
4,Sambo Kojin,Mandaluyong City,"Japanese, Korean",4,1500,4.8


In [10]:
#Create Combined Feature
df['Features'] = (
    df['Cuisines'].astype(str) + " " +
    df['City'].astype(str) + " " +
    df['Price range'].astype(str)
)


In [11]:
#Convert Text into Numbers (TF-IDF)
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['Features'])

In [12]:
#Calculate Similarity
cosine_sim = cosine_similarity(tfidf_matrix)


In [14]:
# Recommendation Function
indices = pd.Series(df.index,
                    index=df['Restaurant Name']).drop_duplicates()

In [18]:
indices = pd.Series(df.index, index=df["Restaurant Name"])

def recommend_restaurant(name, n=5):

    idx = indices[name]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:n+1]

    restaurant_indices = [i[0] for i in similarity_scores]

    return df.iloc[restaurant_indices][[
        "Restaurant Name",
        "City",
        "Cuisines",
        "Average Cost for two",
        "Aggregate rating"
    ]]

In [19]:
recommend_restaurant("Barbeque Nation")

,Restaurant Name,City,Cuisines,Average Cost for two,Aggregate rating
586,Rasoi Ghar,Dubai,Indian,90,4.3
590,Carnival By Tresind,Dubai,Indian,500,4.9
597,Tresind - Nassima Royal Hotel,Dubai,Indian,500,4.9
596,SpiceKlub,Dubai,"Indian, North Indian, Street Food",150,4.4
589,AB's Absolute Barbecues,Dubai,"Continental, Indian",160,4.9


In [21]:
def recommend(cuisine, city, price):

    result = df[
        (df['Cuisines'].str.contains(cuisine, case=False, na=False)) &
        (df['City'].str.contains(city, case=False, na=False)) &
        (df['Price range'] == price)
    ]

    result = result.sort_values(
        by='Aggregate rating',
        ascending=False
    )

    return result[['Restaurant Name',
                   'City',
                   'Cuisines',
                   'Price range',
                   'Average Cost for two',
                   'Aggregate rating']].head(10)

In [22]:
recommend("North Indian", "New Delhi", 3)

,Restaurant Name,City,Cuisines,Price range,Average Cost for two,Aggregate rating
6656,Kopper Kadai,New Delhi,North Indian,3,1400,4.8
3014,Zabardast Indian Kitchen,New Delhi,North Indian,3,1800,4.7
6655,Band Baaja Baaraat,New Delhi,North Indian,3,1300,4.6
6317,Cafe Lota,New Delhi,"North Indian, South Indian, Bihari",3,1200,4.5
6658,Qubitos - The Terrace Cafe,New Delhi,"Thai, European, Mexican, North Indian, Chinese...",3,1500,4.5
6709,Lights Camera Action - Air Bar,New Delhi,"North Indian, Continental",3,1300,4.4
6711,National Highway 44,New Delhi,"Kashmiri, North Indian, Mughlai, South Indian,...",3,1100,4.4
3092,38 Barracks,New Delhi,"North Indian, Italian, Asian, American",3,1600,4.4
6144,Gulati,New Delhi,"North Indian, Mughlai",3,1500,4.4
3770,Chhalava - �__Lava,New Delhi,"North Indian, Italian, Continental",3,1200,4.4


In [25]:
import pickle

In [26]:
pickle.dump(
    tfidf,
    open("tfidf.pkl","wb")
)

pickle.dump(
    cosine_sim,
    open("cosine_similarity.pkl","wb")
)

pickle.dump(
    df,
    open("restaurant_data.pkl","wb")
)

print("Models Saved Successfully!")

Models Saved Successfully!
